In [ ]:
pip install seaborn

In [ ]:
pip install tensorflow


In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import cv2
from glob import glob
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow.keras.preprocessing import image
from matplotlib.image import imread
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.layers import BatchNormalization
from sklearn.metrics import classification_report,confusion_matrix
from tensorflow.keras.models import Sequential, Model
from keras.regularizers import l2
from tensorflow.keras.layers import Activation, Dropout, Dense, Flatten, Conv2D, BatchNormalization, MaxPooling2D, GlobalAveragePooling2D,Input,concatenate, Lambda
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import shutil
import random

# Define dataset paths
folder_path = "D:/sqldev/RetinalOCT_Dataset"

train_dir = os.path.join(folder_path, "D:/sqldev/RetinalOCT_Dataset/train")
val_dir = os.path.join(folder_path, "D:/sqldev/RetinalOCT_Dataset/val")
test_dir = os.path.join(folder_path, "D:/sqldev/RetinalOCT_Dataset/test")





# Create a new directory to store the selected 500 images
selected_data_path = "D:/sqldev/selected_data"
os.makedirs(selected_data_path, exist_ok=True)

# Function to select a limited number of images per class
def select_images(source_dir, dest_dir, num_images=250):
    os.makedirs(dest_dir, exist_ok=True)

    # Iterate over class folders
    for class_name in os.listdir(source_dir):
        class_source_path = os.path.join(source_dir, class_name)
        class_dest_path = os.path.join(dest_dir, class_name)

        if os.path.isdir(class_source_path):
            os.makedirs(class_dest_path, exist_ok=True)

            # List all images in the class
            images = [img for img in os.listdir(class_source_path) if os.path.isfile(os.path.join(class_source_path, img))]

            # Select `num_images` randomly
            selected_images = random.sample(images, min(num_images, len(images)))

            # Copy selected images
            for img in selected_images:
                shutil.copy(os.path.join(class_source_path, img), os.path.join(class_dest_path, img))

# Select 500 images from each dataset (train, val, test)
select_images(train_dir, f"{selected_data_path}/train", num_images=250)
select_images(val_dir, f"{selected_data_path}/val", num_images=250)
select_images(test_dir, f"{selected_data_path}/test", num_images=250)

print("✅ 500 images per dataset selected successfully!")


In [ ]:
pip install torch torchvision matplotlib tqdm


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torchvision import models
import os
import pandas as pd
from torchvision.io import read_image
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time
from tqdm import tqdm


In [ ]:

import torch
from torch.utils.data import Dataset
import os
from torchvision.io import read_image
import torchvision.transforms as T

class CustomImageDataset(Dataset):
    def __init__(self, ds_path, transforms=None, target_transform=None):
        self.ds_path = ds_path
        self.transforms = transforms
        self.target_transform = target_transform
        self.labels = os.listdir(ds_path)  # Get list of label folder names
        self.img_paths = []

        # Create a list of image paths and ensure all labels are mapped correctly
        for label in self.labels:
            base_path = os.path.join(ds_path, label)
            if os.path.isdir(base_path):  # Ensure it's a directory
                for img in os.listdir(base_path):
                    img_path = os.path.join(base_path, img)
                    self.img_paths.append(img_path)

        # Mapping labels to numeric values
        self.label_map = {
            'AMD': 0,
            'CNV': 1,
            'CSR': 2,
            'DME': 3,
            'DR': 4,
            'DRUSEN': 5,
            'MH': 6,
            'NORMAL': 7
        }

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = read_image(img_path)

        # Extract the label from the folder name in a cross-platform way
        label = os.path.basename(os.path.dirname(img_path))

        # Ensure label exists in label_map, otherwise raise an error
        if label not in self.label_map:
            raise KeyError(f"Label '{label}' not found in label_map. Check dataset structure.")

        image = image.float() / 255.0  # Normalize pixel values to [0,1]

        # Convert grayscale images to RGB if needed
        if image.shape[0] == 1:
            image = image.repeat(3, 1, 1)

        # Apply transformations
        if self.transforms:
            image = self.transforms(image)

        image = T.Resize((224, 224))(image)  # Ensure image size is consistent

        return image, self.label_map[label]




In [ ]:
train_path = 'D:/sqldev/selected_data/train'
train_data = CustomImageDataset(train_path)

val_path = 'D:/sqldev/selected_data/val'
val_data = CustomImageDataset(val_path)

test_path = 'D:/sqldev/selected_data/val'
test_data = CustomImageDataset(test_path)

train_len = len(train_data)
train_dataloader = DataLoader(train_data, batch_size=64, shuffle=True)

val_len = len(val_data)
val_dataloader = DataLoader(val_data, batch_size=64, shuffle=True)

test_len = len(test_data)
test_dataloader = DataLoader(test_data, batch_size=8)

data_loaders = {'train': train_dataloader, 'val': val_dataloader}


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device


In [ ]:
def __getitem__(self, idx):
    img_path = self.img_paths[idx]
    image = read_image(img_path)

    # Extract the label correctly for both Windows and Linux
    label = os.path.basename(os.path.dirname(img_path))  # Correct way to get folder name

    image = image.squeeze()
    image = image.repeat(3, 1, 1)
    image = image / 255.0
    if self.transforms:
        image = self.transforms(image)
    if self.target_transform:
        label = self.target_transform(label)

    image = T.Resize((224, 224))(image)

    return image, self.label_map[label]


In [ ]:
class ResNetLSTM(nn.Module):
    def __init__(self, num_classes=8, lstm_hidden_size=256, lstm_num_layers=2):
        super(ResNetLSTM, self).__init__()

        # Load pretrained ResNet18 model
        self.resnet = models.resnet18(pretrained=True)
        # Remove the final fully connected layer
        self.resnet = nn.Sequential(*list(self.resnet.children())[:-1])

        # LSTM to process the features from ResNet
        self.lstm = nn.LSTM(input_size=512,  # The ResNet output size is 512
                            hidden_size=lstm_hidden_size,
                            num_layers=lstm_num_layers,
                            batch_first=True)

        # Fully connected layer for classification
        self.fc = nn.Linear(lstm_hidden_size, num_classes)

    def forward(self, x):
        # Pass through ResNet (without the final classification layer)
        x = self.resnet(x)  # Shape: [batch_size, 512, 1, 1]

        # Flatten ResNet output for LSTM
        x = x.view(x.size(0), -1)  # Shape: [batch_size, 512]
        x = x.unsqueeze(1)  # Adding sequence length dimension: [batch_size, 1, 512]

        # Pass through LSTM
        lstm_out, (hn, cn) = self.lstm(x)  # lstm_out: [batch_size, seq_len, hidden_size]

        # We only need the output from the last LSTM step
        lstm_out = lstm_out[:, -1, :]  # Shape: [batch_size, hidden_size]

        # Final classification
        out = self.fc(lstm_out)  # Shape: [batch_size, num_classes]
        return out


In [ ]:
model = ResNetLSTM(num_classes=8)
model.to(device)

# Criterion and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
def train_model(model, criterion, optimizer, num_epochs=3):
    since = time.time()
    best_acc = 0.0

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch + 1, num_epochs))
        print('-' * 10)
        for mode, data_loader in data_loaders.items():
            if mode == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_correct = 0

            # Iterate over batches
            for inputs, labels in tqdm(data_loader):
                inputs = inputs.to(device)
                labels = labels.to(device)

                with torch.set_grad_enabled(mode == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if mode == 'train':
                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_correct += torch.sum(preds == labels.data)

            data_len = train_len

            if mode == 'val':
                data_len = val_len

            epoch_loss = running_loss / data_len
            epoch_acc = running_correct.double() / data_len
            print('{} loss: {:4f} acc: {:4f}'.format(mode, epoch_loss, epoch_acc))


In [ ]:
train_data = CustomImageDataset('D:/sqldev/selected_data/train')
train_model(model, criterion, optimizer, num_epochs=10)


In [ ]:
def evaluate_model(model, dataloader):
    model.eval()  # Set model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # No gradient calculation for inference
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    accuracy = 100 * correct / total
    print(f"Model Accuracy on Test Data: {accuracy:.2f}%")
    return accuracy


In [ ]:
from sklearn.metrics import classification_report

def classification_analysis(model, dataloader, class_names):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    print("Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
import matplotlib.pyplot as plt

def visualize_predictions(model, dataloader, class_names, num_images=5):
    model.eval()
    images_shown = 0

    fig, axes = plt.subplots(1, num_images, figsize=(15, 5))

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            for i in range(num_images):
                image = images[i].cpu().permute(1, 2, 0).numpy()
                true_label = class_names[labels[i].item()]
                pred_label = class_names[predicted[i].item()]

                axes[i].imshow(image)
                axes[i].set_title(f"Pred: {pred_label}\nTrue: {true_label}")
                axes[i].axis("off")

                images_shown += 1
                if images_shown == num_images:
                    break
            if images_shown == num_images:
                break

    plt.show()


In [ ]:
# Class names (must match label_map in dataset)
class_names = ['AMD', 'CNV', 'CSR', 'DME', 'DR', 'DRUSEN', 'MH', 'NORMAL']

# Evaluate accuracy
evaluate_model(model, test_dataloader)

# Print classification report
classification_analysis(model, test_dataloader, class_names)

# Visualize predictions
visualize_predictions(model, test_dataloader, class_names, num_images=5)


In [ ]:
class_names = ['AMD', 'CNV', 'CSR', 'DME', 'DR', 'DRUSEN', 'MH', 'NORMAL']
plot_confusion_matrix_heatmap(model, test_dataloader, class_names)


In [ ]:
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
def plot_confusion_matrix_heatmap(model, dataloader, class_names):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Normalize the confusion matrix
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    # Plot heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_normalized, annot=True, cmap="Blues", fmt=".2f", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted Labels")
    plt.ylabel("True Labels")
    plt.title("Confusion Matrix Heatmap")
    plt.show(block=True)

